# **XOR Example**
## Setting Up Your First Experiment  

Welcome to Zero2Neuro!  
This notebook is for new users to the toolkit to learn how to get their first experiment running. We'll use XOR to introduce the basics of configuring and training a neural network model with Zero2Neuro.

By the end of this notebook you'll know how to:

- Specify arguments from the command line
- Load arguments from .txt configuration files
- Create a simple deep neural network
- Train a model on a small dataset
- Evaluate the model's predictions

The first thing to do is load up zero2neuro.

In [ ]:
import os
import sys

# Optional if you don't have the neuro path variable set up in your bashrc (Set to folder/directory above keras3_tools and zero2neuro)
# os.environ["NEURO_REPOSITORY_PATH"] = "/home/myuser/neuro"

neuro_path = os.getenv("NEURO_REPOSITORY_PATH")
assert neuro_path is not None, "Environment variable NEURO_REPOSITORY_PATH must be set to directory above zero2neuro and keras3_tools"

sys.path.append(neuro_path + '/zero2neuro/src/')

from zero2neuro import *
from parser import *

In [ ]:
# We create our arguments parser, this is what reads in the arguments and feeds them to the toolkit.
parser = create_parser()

You can declare all arguments in command line but the more concise to do it is to use .txt files that contain the bulk of your arguments. To feed in arguments from .txt files pass them as "@[FILENAME].txt"  
Arguments are read left to right in command line and top to bottom in .txt, if the same argument is read later it will overwrite the older argument call.  
  
Provided in this tutorial folder are three config files, data_config.txt, experiment_config.txt, and network_config.txt. Each of these correspond to different parts of the training process. These files are left with a few blanks but for this first tutorial just follow along and edit it with the values detailed. After getting it up and running it is also beneficial to go back and edit some of the arguments and see how it affects the experiment.
  
# Data Config
Open up data_config.txt, this config file is specifically for loading in the data and telling zero2neuro how to set it up for the model. Inside of the config file should look like this:
  
```
Data is from a table  
--data_format=tabular  
--data_file=[PATH TO "xor_data.csv"]  
  
Our one file becomes our training set  
--data_set_type=fixed  
  
Two inputs  
--data_inputs  
In 0  
In 1  
  
One output  
--data_outputs  
Out 0
```  
The only thing you need to set yourself here is data_file. By default this should be "../xor_data.csv" as the data is one folder up from where this notebook is in. data_format specifies what form the data is coming into the model, for this examples since its csv we put down tabular. We go into more detail about what data_set_type means in the iris example. Our inputs and outputs are our features and the feature we're trying to predict. For this example we take in the two binary inputs which match up to either a 0 or a 1. Our model will use this relationship to try to find a function that maps the relationship between the inputs we specify to achieve the true output. 

Add the data_file argument and you're good to go in data config.  

# Experiment Config
Next we move onto our experiment config, no real need to open this one to edit unless you want to (experiment_config.txt).

```
# Name used for files and wandb
--experiment_name=xor

# Loss function is what we minimize during training; mse=Mean Squared Error
--loss=mse

# We might also care about other metrics; mae=Mean Absolute Error
--metrics
mae
mse

# How fast we adjust the parameters
--learning_rate=0.001

# Maximum number of steps for training
--epochs=5000

# Stop the training early if no improvement is observed
# Note: we typically monitor val_loss when we have a validation data set
--early_stopping
--early_stopping_monitor=loss
--early_stopping_patience=2000

# Where to place trained networks, reports, and stored results
--results_path=./results

# Expanded file name
--output_file_base={args.experiment_name}_R{args.data_rotation:02d}

# Save the trained model to a file
--save_model

# Create a picture of the model
--render_model

# Save the training set results
--log_training_set

# Report the results to a XLSX file
--report
--report_training
--report_training_ins
```
This one is fully filled out but some things to note right now are we can name our experiment (this applies to saved results files), epochs determine the maximum amount of steps the model takes while training, results path is the file path where you want the results from your model placed, by default this will create a results folder in this tutorial directory, output file base is the naming convention for your files, you can save your model to a .keras file (very important if you want to actually utilize your trained model), render model gives you a little diagram of the model architecture, log and report give you .pkl and .xlsx files respectively for your results.  

The rest of the arguments we will go into more detail in later examples so don't worry about them for now.

# Network Config
Finally we have the configuration for the model architecture itself, open up network_config.txt.
```
# Fully connected network
--network_type=fully_connected

# TODO: How many input features are in the dataset
--input_shape
???

# Two hidden layers
# TODO: Experiment with more complex hidden layers
--number_hidden_units
3
2
--hidden_activation=elu

# TODO: How many outputs does the data have
--output_shape
???

# TODO: XOR can predict anywhere between 0 and 1, which output activation reflects that?
--output_activation=???
```

network_type refers to the type of neural network we're using. For this problem we are going to use a fully connect neural network which are great for datasets that lack a spatial or temporal relationship between examples. The actual layers are defined in input_shape, number_hidden_units, and output_shape which maps to input layer, hidden layers, and output layer. Our hidden activation function is what learns complex patterns in the data and passes it along through the network (there's a lot of choices, a popular choice is relu or elu). The output activation is what formats the final prediction of the mdoel. 

We need to set input and output shapes and decide out output activation. Our input shape is essentailly the amount of input features we have coming in, you can figure this out pretty easily by looking at data_config or looking at the .csv file. For output shape it's the same concept but for how many output features you're predicting, which can be found out the smae way.  

Output activation is our final formatting, so think about what our possible predictions are by looking at the data. In this data set we are predicting either 0 or 1, so we want to constrain our prediction into that space which means an output activation like sigmoid is most appropiate as it forces that 0 to 1 range.  

Once you fill out these missing arguments you should have a fully filled out group of config files. We can now run your first experiment!



In [ ]:
# We pass in our config files, -v is verbosity (-vvv will visualize each trainig step, since our epochs are so high its best to keep this low). --force will overwrite previous experiments with the same name.
arg_string = "@network_config.txt @data_config.txt @experiment_config.txt -v --force"
args = parser.parse_args(arg_string.split())
print(args)

In [ ]:
# Pass arguments into the toolkit and train
prepare_and_execute_experiment(args)

# Examining Results
Congratulations on training your first model, if you navigate to your results folder you will find four things: your trained .keras model, a .png of the architecture, a .pkl files, and a .xlsx files. To view the .xlsx file just download it and open it up with the software of your choice. In this notebook we are going to use the .pkl file to make some observations and visualizations.  

pkl files are great for examining with python, let's write some code and look at it.

In [ ]:
# We open up our pkl file and load in the data
with open('results/xor_R00_results.pkl', 'rb') as pickle_file:
    data = pickle.load(pickle_file) # Grab the data from pickle

In [ ]:
# Zero2Neuro uses pkl files to store a dictionary, here we look at the keys. 
print(data.keys())

In [ ]:
# We can write some code to look at the predictions directly, it's a very small dataset so it's easy to examine
print('Training Data\n', data['ins_training'])
print('Output Data\n', data['outs_training'])
print('What the model predicted\n', data['predict_training'])

Your predictions should be close to the output data but won't be exactly 0 or 1. For instance when predicting a value of 0 a prediction of 0.0x is expected. Let's look to see how the model did in training.

In [ ]:
plt.plot(data['history']['loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Epoch/Loss')
plt.grid(True)
plt.show()

XOR is an interesting case as it stays flat for a little while (roughly 200 epochs with the given setup) before learning. This epoch/loss graph shows where the loss is at per epoch, we want to minimize our loss (a perfect loss is 0, although this is not realistic and in some cases actually dangerous). 

This concludes the XOR example. If you're going through example by example the next step would be our breast_cancer example so navigate to that tutorial folder.